<a href="https://colab.research.google.com/github/Johnson-Prosper/Detecteur-Dechets-Streamlit/blob/main/Detecteur_dechets_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install streamlit opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 86.1 MB/s eta 0:00:00


In [6]:
import os
from datetime import datetime

import cv2
import numpy as np
import streamlit as st
import tensorflow.lite as tflite

from PIL import Image, ImageOps


# ============================================================
# 1. CONFIGURATION GÉNÉRALE
# ============================================================

st.set_page_config(
    page_title="EcoSort AI",
    page_icon="♻️",
    layout="centered",
    initial_sidebar_state="collapsed"
)


# ============================================================
# 2. STYLE DE L'APPLICATION
# ============================================================

st.markdown(
    """
    <style>

    .main {
        padding-top: 1rem;
    }

    .eco-title {
        text-align: center;
        font-size: 2.2rem;
        font-weight: 800;
        margin-bottom: 0.2rem;
    }

    .eco-subtitle {
        text-align: center;
        color: #666;
        margin-bottom: 1.5rem;
    }

    .result-card {
        padding: 1.5rem;
        border-radius: 20px;
        background: #f7f7f7;
        text-align: center;
        margin-top: 1rem;
        margin-bottom: 1rem;
    }

    .result-category {
        font-size: 1.8rem;
        font-weight: 800;
        margin: 0.5rem 0;
    }

    .confidence {
        font-size: 1.1rem;
        margin-bottom: 0.5rem;
    }

    .info-card {
        padding: 1.2rem;
        border-radius: 16px;
        background: #f5f5f5;
        margin-top: 1rem;
    }

    .metric-card {
        padding: 1rem;
        border-radius: 16px;
        background: #f5f5f5;
        text-align: center;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# 3. TITRE
# ============================================================

st.markdown(
    '<div class="eco-title">♻️ EcoSort AI</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="eco-subtitle">'
    'Analyse intelligente et aide au tri des déchets'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# 4. CONFIGURATION DU MODÈLE
# ============================================================

MODEL_PATH = "model_unquant.tflite"

# IMPORTANT :
# Cet ordre doit correspondre EXACTEMENT à celui utilisé
# pendant l'entraînement du modèle.
CLASSES = [
    "Non-Recyclable",
    "Organique",
    "Recyclable"
]

# Seuil minimal pour accepter une prédiction
CONFIDENCE_THRESHOLD = 0.60

# Seuil de qualité d'image
BLUR_THRESHOLD = 40
BRIGHTNESS_MIN = 35
BRIGHTNESS_MAX = 225


# ============================================================
# 5. BASE DE CONNAISSANCES
# ============================================================

GUIDE_TRI = {

    "Non-Recyclable": {
        "emoji": "🗑️",
        "titre": "NON-RECYCLABLE",
        "couleur": "orange",
        "description": (
            "L'intelligence artificielle classe cet objet "
            "dans la catégorie des déchets non recyclables."
        ),
        "action": (
            "Placez-le dans le conteneur prévu pour les déchets "
            "non recyclables, selon les règles locales."
        ),
        "conseil": (
            "Lorsque c'est possible, vérifiez les consignes "
            "locales avant de jeter l'objet."
        )
    },

    "Organique": {
        "emoji": "🌱",
        "titre": "ORGANIQUE",
        "couleur": "green",
        "description": (
            "L'objet semble appartenir à la catégorie des "
            "déchets organiques."
        ),
        "action": (
            "Orientez-le vers la filière organique ou compostable "
            "si cette filière existe dans votre zone."
        ),
        "conseil": (
            "Évitez de mélanger les déchets organiques avec "
            "les déchets recyclables."
        )
    },

    "Recyclable": {
        "emoji": "♻️",
        "titre": "RECYCLABLE",
        "couleur": "blue",
        "description": (
            "L'intelligence artificielle considère cet objet "
            "comme appartenant à la catégorie recyclable."
        ),
        "action": (
            "Placez-le dans la filière destinée aux déchets "
            "recyclables lorsque cette filière est disponible."
        ),
        "conseil": (
            "Lorsque c'est pertinent, videz et nettoyez l'objet "
            "avant de le déposer."
        )
    }
}


# ============================================================
# 6. CHARGEMENT DU MODÈLE
# ============================================================

@st.cache_resource
def charger_modele():

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Le fichier '{MODEL_PATH}' est introuvable."
        )

    interpreter = tflite.Interpreter(
        model_path=MODEL_PATH
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    return interpreter, input_details, output_details


# ============================================================
# 7. INITIALISATION DE L'HISTORIQUE
# ============================================================

if "historique" not in st.session_state:
    st.session_state.historique = []


# ============================================================
# 8. OUTILS IMAGE
# ============================================================

def corriger_orientation(image):
    """
    Corrige l'orientation EXIF des photos prises avec un téléphone.
    """
    return ImageOps.exif_transpose(image)


def analyser_qualite_image(image):
    """
    Analyse approximativement :
    - luminosité
    - netteté/flou
    """

    image_rgb = np.array(image.convert("RGB"))

    gray = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2GRAY
    )

    brightness = float(np.mean(gray))

    blur_score = float(
        cv2.Laplacian(gray, cv2.CV_64F).var()
    )

    messages = []
    image_correcte = True

    if brightness < BRIGHTNESS_MIN:
        messages.append(
            "📷 L'image semble trop sombre."
        )
        image_correcte = False

    elif brightness > BRIGHTNESS_MAX:
        messages.append(
            "☀️ L'image semble très lumineuse."
        )

    if blur_score < BLUR_THRESHOLD:
        messages.append(
            "🔎 L'image semble légèrement floue."
        )
        image_correcte = False

    return {
        "brightness": brightness,
        "blur_score": blur_score,
        "messages": messages,
        "correcte": image_correcte
    }


# ============================================================
# 9. PRÉPARATION DE L'IMAGE POUR L'IA
# ============================================================

def preparer_image(image, input_details):

    input_shape = input_details[0]["shape"]
    input_dtype = input_details[0]["dtype"]

    # Récupération automatique de la taille
    hauteur = int(input_shape[1])
    largeur = int(input_shape[2])

    image = corriger_orientation(image)
    image = image.convert("RGB")
    image = image.resize((largeur, hauteur))

    image_array = np.array(image)

    # --------------------------------------------------------
    # Modèle FLOAT
    # --------------------------------------------------------

    if input_dtype == np.float32:

        image_array = image_array.astype(np.float32)

        image_array = image_array / 255.0

    # --------------------------------------------------------
    # Modèle UINT8 / INT8
    # --------------------------------------------------------

    else:

        image_array = image_array.astype(input_dtype)

        quantization = input_details[0].get(
            "quantization",
            (0.0, 0)
        )

        scale, zero_point = quantization

        if scale != 0:

            image_array = (
                image_array.astype(np.float32) / scale
                + zero_point
            )

            image_array = np.clip(
                image_array,
                np.iinfo(input_dtype).min,
                np.iinfo(input_dtype).max
            ).astype(input_dtype)

    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    return image_array


# ============================================================
# 10. SORTIE DU MODÈLE
# ============================================================

def obtenir_predictions(
    interpreter,
    input_details,
    output_details,
    image_tensor
):

    interpreter.set_tensor(
        input_details[0]["index"],
        image_tensor
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]["index"]
    )[0]

    output = np.array(
        output,
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Si le modèle renvoie des logits plutôt que des
    # probabilités, transformation softmax.
    # --------------------------------------------------------

    if (
        np.min(output) < 0
        or np.max(output) > 1
        or not np.isclose(np.sum(output), 1.0, atol=0.05)
    ):

        exp_values = np.exp(
            output - np.max(output)
        )

        output = (
            exp_values
            / np.sum(exp_values)
        )

    return output


# ============================================================
# 11. ANALYSE PRINCIPALE
# ============================================================

def analyser_dechet(image, interpreter, input_details, output_details):

    image_tensor = preparer_image(
        image,
        input_details
    )

    predictions = obtenir_predictions(
        interpreter,
        input_details,
        output_details,
        image_tensor
    )

    # Sécurité : vérifier la correspondance
    if len(predictions) != len(CLASSES):

        raise ValueError(
            "Le nombre de sorties du modèle "
            f"({len(predictions)}) ne correspond pas "
            f"au nombre de classes configurées "
            f"({len(CLASSES)})."
        )

    meilleur_index = int(
        np.argmax(predictions)
    )

    classe_detectee = CLASSES[
        meilleur_index
    ]

    score_confiance = float(
        predictions[meilleur_index]
    )

    # Classement de toutes les prédictions
    classement = sorted(
        zip(CLASSES, predictions),
        key=lambda x: x[1],
        reverse=True
    )

    return (
        classe_detectee,
        score_confiance,
        classement
    )


# ============================================================
# 12. ENREGISTRER DANS L'HISTORIQUE
# ============================================================

def ajouter_historique(
    classe,
    confiance
):

    maintenant = datetime.now()

    entree = {
        "date": maintenant.strftime("%d/%m/%Y"),
        "heure": maintenant.strftime("%H:%M"),
        "classe": classe,
        "confiance": confiance
    }

    st.session_state.historique.insert(
        0,
        entree
    )

    # On conserve les 50 dernières analyses
    st.session_state.historique = (
        st.session_state.historique[:50]
    )


# ============================================================
# 13. CHARGEMENT
# ============================================================

try:

    (
        interpreter,
        input_details,
        output_details
    ) = charger_modele()

except FileNotFoundError as e:

    st.error(f"❌ {e}")

    st.info(
        "Placez 'model_unquant.tflite' "
        "dans le même dossier que votre application."
    )

    st.stop()

except Exception as e:

    st.error(
        "❌ Impossible de charger le modèle."
    )

    st.exception(e)

    st.stop()


# ============================================================
# 14. INFORMATIONS TECHNIQUES
# ============================================================

with st.expander("🔧 Informations techniques du modèle"):

    input_shape = input_details[0]["shape"]
    input_dtype = input_details[0]["dtype"]

    st.write(
        f"**Entrée :** {list(input_shape)}"
    )

    st.write(
        f"**Type :** `{input_dtype}`"
    )

    st.write(
        f"**Nombre de classes :** {len(CLASSES)}"
    )

    st.write(
        "**Classes :** "
        + ", ".join(CLASSES)
    )


# ============================================================
# 15. CHOIX DE LA SOURCE
# ============================================================

st.subheader("📷 Analyser un déchet")

source = st.radio(
    "Choisissez la méthode :",
    [
        "📸 Prendre une photo",
        "🖼️ Importer une image"
    ],
    horizontal=True
)

img_file = None


if source == "📸 Prendre une photo":

    img_file = st.camera_input(
        "Photographiez le déchet"
    )

else:

    img_file = st.file_uploader(
        "Choisissez une image",
        type=[
            "png",
            "jpg",
            "jpeg",
            "webp"
        ]
    )


# ============================================================
# 16. ANALYSE DE L'IMAGE
# ============================================================

if img_file is not None:

    try:

        image = Image.open(img_file)

        image = corriger_orientation(image)

        st.image(
            image,
            caption="Image sélectionnée",
            use_container_width=True
        )

        # ----------------------------------------------------
        # QUALITÉ
        # ----------------------------------------------------

        qualite = analyser_qualite_image(
            image
        )

        if qualite["messages"]:

            for message in qualite["messages"]:
                st.warning(message)

        # ----------------------------------------------------
        # ANALYSE IA
        # ----------------------------------------------------

        with st.spinner(
            "🧠 Analyse du déchet..."
        ):

            (
                classe_detectee,
                score_confiance,
                classement
            ) = analyser_dechet(
                image,
                interpreter,
                input_details,
                output_details
            )

        # ----------------------------------------------------
        # CONFIANCE INSUFFISANTE
        # ----------------------------------------------------

        if score_confiance < CONFIDENCE_THRESHOLD:

            st.warning(
                "⚠️ L'IA n'est pas suffisamment certaine "
                "de sa réponse."
            )

            st.write(
                f"Confiance maximale : "
                f"**{score_confiance * 100:.1f} %**"
            )

            st.info(
                "Essayez avec une photo plus rapprochée, "
                "plus nette et mieux éclairée."
            )

        else:

            # ------------------------------------------------
            # RÉSULTAT
            # ------------------------------------------------

            informations = GUIDE_TRI.get(
                classe_detectee
            )

            st.markdown(
                f"""
                <div class="result-card">

                    <div style="font-size:3rem;">
                        {informations["emoji"]}
                    </div>

                    <div class="result-category">
                        {informations["titre"]}
                    </div>

                    <div class="confidence">
                        Confiance de l'IA :
                        <strong>
                            {score_confiance * 100:.1f} %
                        </strong>
                    </div>

                </div>
                """,
                unsafe_allow_html=True
            )

            # ------------------------------------------------
            # BARRE DE CONFIANCE
            # ------------------------------------------------

            st.progress(
                min(score_confiance, 1.0)
            )

            # ------------------------------------------------
            # EXPLICATION
            # ------------------------------------------------

            st.subheader("🔎 Pourquoi ?")

            st.write(
                informations["description"]
            )

            # ------------------------------------------------
            # ACTION
            # ------------------------------------------------

            st.subheader("♻️ Que faire ?")

            st.info(
                informations["action"]
            )

            # ------------------------------------------------
            # CONSEIL
            # ------------------------------------------------

            st.subheader("💡 Conseil")

            st.success(
                informations["conseil"]
            )

            # ------------------------------------------------
            # AUTRES PRÉDICTIONS
            # ------------------------------------------------

            with st.expander(
                "📊 Voir les autres prédictions"
            ):

                for classe, score in classement:

                    st.write(
                        f"**{classe}** — "
                        f"{score * 100:.1f} %"
                    )

                    st.progress(
                        min(float(score), 1.0)
                    )

            # ------------------------------------------------
            # HISTORIQUE
            # ------------------------------------------------

            ajouter_historique(
                classe_detectee,
                score_confiance
            )

    except Exception as e:

        st.error(f"❌ Une erreur est survenue lors du traitement de l'image : {e}")


# ============================================================
# 17. HISTORIQUE ET STATISTIQUES
# ============================================================

st.divider()

st.subheader("📊 Mes statistiques")

historique = st.session_state.historique


if not historique:

    st.info(
        "Aucune analyse réalisée pour le moment."
    )

else:

    total = len(historique)

    recyclables = sum(
        1
        for x in historique
        if x["classe"] == "Recyclable"
    )

    organiques = sum(
        1
        for x in historique
        if x["classe"] == "Organique"
    )

    non_recyclables = sum(
        1
        for x in historique
        if x["classe"] == "Non-Recyclable"
    )

    col1, col2, col3 = st.columns(3)

    with col1:

        st.metric(
            "🔎 Analyses",
            total
        )

    with col2:

        st.metric(
            "♻️ Recyclables",
            recyclables
        )

    with col3:

        st.metric(
            "🌱 Organiques",
            organiques
        )

    st.metric(
        "🗑️ Non-recyclables",
        non_recyclables
    )


# ============================================================
# 18. HISTORIQUE DÉTAILLÉ
# ============================================================

with st.expander("🕒 Voir l'historique"):

    if not historique:

        st.write(
            "Aucune analyse."
        )

    else:

        for entree in historique:

            informations = GUIDE_TRI.get(
                entree["classe"]
            )

            st.write(
                f"{informations['emoji']} "
                f"**{entree['classe']}** — "
                f"{entree['confiance'] * 100:.1f}%"
            )

            st.caption(
                f"{entree['date']} à {entree['heure']}"
            )

            st.divider()


# ============================================================
# 19. GUIDE DU TRI
# ============================================================

with st.expander("📚 Guide des catégories"):

    for classe, informations in GUIDE_TRI.items():

        st.markdown(
            f"### {informations['emoji']} "
            f"{informations['titre']}"
        )

        st.write(
            informations["description"]
        )

        st.write(
            f"**Action :** {informations['action']}"
        )

        st.divider()


# ============================================================
# 20. RÉINITIALISATION
# ============================================================

if historique:

    if st.button(
        "🗑️ Effacer l'historique"
    ):

        st.session_state.historique = []

        st.rerun()


# ============================================================
# 21. AVERTISSEMENT
# ============================================================

st.caption(
    "⚠️ EcoSort AI fournit une estimation basée sur "
    "un modèle d'intelligence artificielle. "
    "Les règles réelles de tri peuvent varier selon "
    "la ville, le pays et les infrastructures locales."
)

2026-10-05 16:48:27.966 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:48:27.971 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:48:28.760 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-05 16:48:28.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:48:28.763 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:48:28.764 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-05 16:48:28.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

NameError: name 'input_details' is not defined